### calculate pass@k

In [ ]:
import numpy as np

In [32]:
def pass_at_k(n, c, k):
    """
    :param n: total number of samples
    :param c: number of correct samples
    :param k: k in pass@$k$
    """
    if n - c < k: return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

In [36]:
n = 10
c = 6
k = 3
pass_at_k(n, c, k)

np.float64(0.9666666666666667)

### calculate functional correctness, codebleu, runtime

In [21]:
from datasets import load_dataset
dataset = load_dataset("openai_humaneval")

c:\Users\tansz\sc4052\tdd-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
import random

random.seed(42) 
all_data = list(dataset["test"])
selected_data = random.sample(all_data, 60)

In [23]:
from codebleu import calc_codebleu
import tempfile
import subprocess
import csv
import black
import time
from tqdm import tqdm

In [ ]:
import argparse
import os
import sys
from dotenv import load_dotenv
from agent import Agent
from tdd_agent import TDD_Agent
from tdd_multiagent import TDD_MultiAgent
from providers import ClaudeProvider, GroqProvider
from tools import ReadFileTool, WriteFileTool, ShellTool

load_dotenv()
api_key = os.environ.get("GROQ_API_KEY")
provider = GroqProvider(api_key=api_key, model="openai/gpt-oss-120b")
tools = [
        ReadFileTool(),
        WriteFileTool(),
        ShellTool(),
    ]

In [28]:
which = "multi-agent"
dir = "eval-multi"
# agent = Agent(provider=provider, tools=tools, directory=dir)
agent = TDD_MultiAgent(test_provider=provider, code_provider=provider, tools=tools, directory=dir, transcript_path=None)

In [ ]:
csv_file = os.path.join(dir, "test_results.csv")
with open(csv_file, "a", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["functional correctness", "codebleu", "runtime"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for data in tqdm(selected_data):

        start_time = time.time()
        agent.run("write a file eval.py with the function: " + data["prompt"])
        end_time = time.time()
        runtime = end_time - start_time

        with open(os.path.join(dir, "eval.py"), "r") as f:
            code = f.read()

        f = tempfile.NamedTemporaryFile("w", suffix=".py", dir=dir, delete=False)
        f.write(data["test"])
        f.write("\n\n")
        f.write(code)
        f.write("\n\n")
        f.write(f"check({data['entry_point']})")
        test_file = f.name
        os.chmod(test_file, 0o777)
        f.close()

        try:
            result = subprocess.run(
                ["python", test_file],
                capture_output=True,
                text=True,
                check=True
            )
            # passed
            score = 1

        except subprocess.CalledProcessError as e:
            # failed
            score = 0
        
        os.remove(test_file)

        pred = code.split("\"\"\"")[-1]
        try:
            pred = black.format_str(pred, mode=black.Mode())
        except:
            pass
        predictions = [pred]
        references = [[data['canonical_solution']]]

        result = calc_codebleu(
            references,
            predictions,
            lang="python"
        )
        b = result["codebleu"]

        writer.writerow({"functional correctness": score, "codebleu": b, "runtime": runtime})

        time.sleep(60)  # to avoid rate limit


